## 1. On charge et on lit notre csv en precisant que la separation des colonnes est : ; et non ,

In [ ]:
import pandas as pd
import os 
df = pd.read_csv("project_dataset.csv", sep=';')

## 2. On inspecte toutes les informations et les valeurs qui sont nulles 

In [ ]:
df.info()
df.describe()
df.isnull().sum()

## 3. On remarque que 90 % des commentaires sont nulls donc on drop les colonnes car elles ne servent a rien pour un modele de prediction

In [ ]:
df.drop(columns=["Commentaire annulations","Commentaire retards au départ","Commentaire retards à l'arrivée"], inplace=True)

## 4. On supprime les duplicates

In [ ]:
df.drop_duplicates(inplace=True)

## 5. On remarque que ~ 2% des valeurs sont nulles 240 pour 12000 entrees donc on les supprimes pas mais on les remplace par leur mediane respective

In [ ]:
for col in df.select_dtypes(include="number").columns: 
    df[col].fillna(df[col].median(), inplace=True)

## 6. On remplace les types des donnees par un type plus approprié

In [ ]:
df["Date"] = pd.to_datetime(df["Date"],format="%Y-%m")

numeric_cols = [
    "Durée moyenne du trajet", "Nombre de circulations prévues", "Nombre de trains annulés",
    "Nombre de trains en retard au départ", "Retard moyen des trains en retard au départ",
    "Retard moyen de tous les trains au départ", "Nombre de trains en retard à l'arrivée",
    "Retard moyen des trains en retard à l'arrivée", "Retard moyen de tous les trains à l'arrivée",
    "Nombre trains en retard > 15min", "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
    "Nombre trains en retard > 30min", "Nombre trains en retard > 60min",
    "Prct retard pour causes externes","Prct retard pour cause infrastructure",
    "Prct retard pour cause gestion trafic","Prct retard pour cause matériel roulant",
    "Prct retard pour cause gestion en gare et réutilisation de matériel",
    "Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
]

for col in numeric_cols:
    df[col] = df[col].str.replace(",", ".", regex=False)  # certaines valeurs sont avec des , et non des . donc on change 
    df[col] = pd.to_numeric(df[col], errors="coerce")    # on convertit les valeurs object en numeriques 

categorical_cols = ["Service", "Gare de départ", "Gare d'arrivée"]
for col in categorical_cols:
    df[col] = df[col].astype("category") # on les convertit en category car pas vraiment utile


## 7. On remplace les valuers NaN dans les types category que nous avons cree juste avant par mode() qui est la valeur la plus presente de la colonne

In [ ]:
cat_cols = df.select_dtypes(include="category").columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

## 8. On fait les features avec la date pour que ce soit plus pratique pour notre modele

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df.drop(columns=["Date"], inplace=True) # on surppimes les dates plus utiles 

## 9. on extrait notre fichier trié en csv